In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "fake_or_proof").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, models, transforms

# Image Resize

In [ ]:
IMG_SIZE = 224

transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]
)

# Loading images 

In [ ]:
def safe_loader(path):
    try:
        img = Image.open(path)
        if img.mode == "P":
            img = img.convert("RGBA")
        img = img.convert("RGB")
        return img
    except Exception as e:
        print(f"Skipping corrupted image: {path}")
        return None  # Skip corrupt images

In [ ]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, fake, real, transform=None):
        self.data = []
        self.transform = transform
        DATASET = Path("../dataset/complete_set")
        DATASET_REAL_DIR = DATASET / "real"
        DATASET_FAKE_DIR = DATASET / "fake"

        data_fake = pd.read_csv(fake)
        data_real = pd.read_csv(real)
        for _, row in data_fake.iterrows():
            image_path = os.path.join(DATASET_FAKE_DIR, row.iloc[0])
            self.data.append((image_path, 0))
        for _, row in data_real.iterrows():
            image_path = os.path.join(DATASET_REAL_DIR, row.iloc[0])
            self.data.append((image_path, 1))
        random.shuffle(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = safe_loader(img_path)
        if self.transform:
            img = self.transform(img)
        return img, label, img_path


PATH_CSV_FAKE = Path("../dataset/fake_images.csv")
PATH_CSV_REAL = Path("../dataset/real_images.csv")
dataset = CustomDataset(PATH_CSV_FAKE, PATH_CSV_REAL, transform=transform)
print(f"Dataset size: {len(dataset)}")

# Splitting the subset into training set and validation set and test set

In [ ]:
size = len(dataset)

train_size = int(0.7 * size)
val_size = int(0.15 * size)
test_size = size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42),
)

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, num_workers=0
)

In [ ]:
import os
import shutil


def save_test_images(test_dataset, output_dir="test_folder"):
    os.makedirs(output_dir, exist_ok=True)
    for i in range(len(test_dataset)):
        _, label, img_path = test_dataset[i]
        filename = os.path.basename(img_path)
        dest_path = os.path.join(output_dir, f"{label}_{filename}")
        shutil.copy(img_path, dest_path)

    print(f"{len(test_dataset)} files copy to{output_dir}")


save_test_images(test_dataset, output_dir="test_folder")

In [ ]:
model = models.resnet18(weights=True)
model.fc = nn.Linear(model.fc.in_features, 2)

# Training the model

In [ ]:
if torch.cuda.is_available():
    print("Using GPU")
else:
    print("Using CPU")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader)}"
    )

# Evalutation of the model on validation set

In [ ]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels, _ in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

In [ ]:
from fake_or_proof.model import checkpoint_for

torch.save(checkpoint_for(model), "notebook_candidate.pth")
print("Restricted checkpoint saved successfully!")

In [ ]:
from fake_or_proof.model import load_checkpoint

model = load_checkpoint("notebook_candidate.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader)}"
    )

In [ ]:
def evaluation(loader, testing_model):
    correct = 0
    total = 0
    testing_model.eval()
    with torch.no_grad():
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = testing_model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Accuracy: {100 * correct / total:.2f}%")


evaluation(val_loader, model)

In [ ]:
from fake_or_proof.model import checkpoint_for

torch.save(checkpoint_for(model), "notebook_candidate.pth")
print("Restricted checkpoint saved successfully!")

In [ ]:
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2

model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10


def train(num_epochs):
    train_avg_loss = []
    test_avg_loss = []
    test_accuracy = []

    for epoch in range(num_epochs):
        model.train()
        train_losses = []

        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_avg_loss.append(sum(train_losses) / len(train_losses))
        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Train Loss: {train_avg_loss[-1]}"
        )

        model.eval()
        test_losses = []
        correct = 0
        total = 0

        with torch.no_grad():
            for x, y, _ in val_loader:
                x, y = x.to(device), y.to(device)

                pred = model(x)
                loss = criterion(pred, y)
                test_losses.append(loss.item())

                _, predicted = torch.max(pred.data, 1)
                total += y.size(0)
                correct += (predicted == y).sum().item()

        test_avg_loss.append(sum(test_losses) / len(test_losses))
        test_accuracy.append(correct / total)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}], "
            f"Test Loss: {test_avg_loss[-1]}, "
            f"Test Accuracy: {test_accuracy[-1]}"
        )

    return train_avg_loss, test_avg_loss, test_accuracy


train_avg_loss, test_avg_loss, test_accuracy = train(10)

In [ ]:
# Plotting the statistics
plt.figure(figsize=(12, 5))

# Plot training and testing losses
plt.subplot(1, 2, 1)
plt.plot(range(1, 10 + 1), train_avg_loss, label="Training Loss")
plt.plot(range(1, 10 + 1), test_avg_loss, label="Testing Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Testing Losses")
plt.legend()

# Plot testing accuracy
plt.subplot(1, 2, 2)
plt.plot(
    range(1, 10 + 1), test_accuracy, label="Testing Accuracy", color="orange"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Testing Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from fake_or_proof.model import checkpoint_for

torch.save(checkpoint_for(model), "notebook_candidate.pth")
print("Restricted checkpoint saved successfully!")

In [ ]:
from fake_or_proof.model import checkpoint_for, load_checkpoint

model = load_checkpoint("notebook_candidate.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels, _ in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs}, Loss: {running_loss / len(train_loader)}"
    )

torch.save(checkpoint_for(model), "notebook_candidate.pth")
print("Model saved.")
evaluation(val_loader, model)

In [ ]:
from fake_or_proof.model import load_checkpoint

model = load_checkpoint("notebook_candidate.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

evaluation(test_loader, model)

# Manual test

In [ ]:
model.eval()
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        for i in range(len(images)):
            img = images[i].cpu().numpy()
            img = img * 0.5 + 0.5
            img = np.transpose(img, (1, 2, 0))

            plt.imshow(img)
            plt.title(
                f"Predicted: {preds[i].item()}, True: {labels[i].item()}"
            )
            plt.axis("off")
            plt.show()
        break

We took 8 human images to test if the model returns the right answer. 

# Saving the model

# Confusion matrix

In [ ]:
model.eval()

# Stores predicitions and labels
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

class_names = ["AI-Generated", "Human-Made"]

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
f1 = 2 * (precision * sensitivity) / (precision + sensitivity)
print(f"Sensitivity: {sensitivity:.2f}")
print(f"Specificity: {specificity:.2f}")
print(f"Precision: {precision:.2f}")
print(f"F1 Score: {f1:.2f}")

Specificity is the proportion of correctly classified negatives among all negative values

Sensitivity(recall) is the proportion of correctly classified positives among all positive values

Precision is the proportion of actual positives among all the predicted positives

F-measure offers a good trade-off between recall and precision if they are both important 

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

fpr, tpr, _ = roc_curve(all_labels, all_preds)
roc_auc = roc_auc_score(all_labels, all_preds)

plt.figure(figsize=(6, 5))
plt.plot(
    fpr,
    tpr,
    color="darkorange",
    lw=2,
    label=f"ROC curve (area = {roc_auc:.2f})",
)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()